# Этап 11 V1 — RealMLP против GBDT_mean

## Исследовательский вопрос

Может ли преднастроенный RealMLP на тех же 47 разрешённых признаках дать существенное улучшение относительно принятого B*=GBDT_mean?

Проверяется ровно одна модель: официальный PyTabKit RealMLP_TD_Classifier. Это официальная преднастроенная конфигурация, а не HPO, не ensemble и не AutoGluon wrapper.

Неизменны dataset Data_final.xlsb, target DefMark, identifier INN, рабочая выборка из 289614 строк, порядок строк, 47 признаков, outer StratifiedKFold(3, shuffle=True, random_state=42), seeds 43/44/45 и final test. Q_B1_norm и Q_B2_norm не являются predictors. GBDT заново не обучается: baseline берётся из сохранённого leakage-safe Stage 7 OOF.

Параметры, фиксируемые для контролируемого outer-fold protocol: device=cpu, n_cv=1, n_refit=0, n_ens=1 и fold-specific random_state. Никаких HPO, bagging/ensembling, calibration, class weighting, balancing, sampling или threshold optimization нет. Встроенная предобработка и внутренняя validation/early stopping RealMLP обучаются только на соответствующем outer-train fold.

Основная метрика выбора — полный OOF Gini. Precision, Recall и F1 при 0.5 являются только диагностикой. Правило решения зафиксировано до запуска: material_gain, если ΔGini >= +0.010 и RealMLP выигрывает Gini минимум на 2/3 folds; inferior, если ΔGini <= -0.010 и проигрывает минимум на 2/3; иначе no_material_benefit.


### Что проверяем?

Подготавливаем воспроизводимый contract: пути, feature identity, hashes, CV и выходные artifacts. Это нужно сделать до чтения данных и до обучения, чтобы experiment не мог молча изменить locked design. Неизменными остаются все Stage 1–10 artifacts.


## Почему проверяем это сейчас?

К моменту Stage 11 основная цепочка Stage 1–10 уже показала, что несколько альтернативных моделей и способов их комбинации не дали существенного преимущества над `GBDT_mean`.

RealMLP запускался независимо и поэтому полезен как дополнительная проверка этого вывода на ещё одном современном подходе к табличным данным.

Здесь не ищется новая конфигурация и не проводится tuning. Проверяется один заранее выбранный официальный tuned-default RealMLP на том же 47-признаковом протоколе.

Если RealMLP покажет material gain, это станет новым evidence против предположения о текущем model-family ceiling.

Если существенного улучшения не будет, результат усилит уже накопленную картину: простая смена архитектуры на тех же 47 признаках пока не устраняет наблюдаемый разрыв качества.

In [1]:
from __future__ import annotations

import copy
import hashlib
import importlib.metadata
import json
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pytabkit import RealMLP_TD_Classifier
from pytabkit.models.sklearn.default_params import DefaultParams
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
GENERATED, SUMMARY = ROOT / 'reports' / 'generated', ROOT / 'reports' / 'summary'
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
STAGE1_PATH = GENERATED / 'stage1_baseline_results_V2.json'
STAGE7_PATH = GENERATED / 'stage7_tabm_stacking_results_V1.json'
STAGE7_OOF_PATH = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
RESULT_PATH = GENERATED / 'stage11_realmlp_results_V1.json'
OOF_PATH = GENERATED / 'stage11_realmlp_oof_V1.npz'
SUMMARY_PATH = SUMMARY / 'stage11_realmlp_summary_V1.json'

TARGET, IDENTIFIER = 'DefMark', 'INN'
FORBIDDEN = ('Q_B1_norm', 'Q_B2_norm')
EXPECTED_DATASET_SHA = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_SHA = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_N, OUTER_SEED, FOLD_SEEDS, THRESHOLD = 289614, 42, (43, 44, 45), 0.5
EXPERIMENT_OVERRIDES = {'device': 'cpu', 'n_cv': 1, 'n_refit': 0, 'n_ens': 1, 'verbosity': 2}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_indices(indices: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()

def metrics_at_threshold(target: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= THRESHOLD).astype(np.int8)
    auc = float(roc_auc_score(target, probability))
    return {'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0, 'PR-AUC': float(average_precision_score(target, probability)), 'Precision': float(precision_score(target, predicted, zero_division=0)), 'Recall': float(recall_score(target, predicted, zero_division=0)), 'F1': float(f1_score(target, predicted, zero_division=0))}

def json_safe(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return repr(value)

OFFICIAL_REALMLP_TD_CLASS_DEFAULTS = json_safe(copy.deepcopy(DefaultParams().RealMLP_TD_CLASS))

def effective_realmlp_config(seed: int) -> dict:
    fold_overrides = {**EXPERIMENT_OVERRIDES, 'random_state': seed}
    return {**copy.deepcopy(OFFICIAL_REALMLP_TD_CLASS_DEFAULTS), **fold_overrides}

def atomic_json(path: Path, payload: dict) -> None:
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False, suffix='.tmp') as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
    os.replace(stream.name, path)

def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    with tempfile.NamedTemporaryFile('wb', dir=path.parent, delete=False, suffix='.tmp') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(stream.name, path)

print(f'PyTabKit: {importlib.metadata.version("pytabkit")}; устройство: cpu')
print(f'Выходные artifacts: {RESULT_PATH.name}, {OOF_PATH.name}, {SUMMARY_PATH.name}')
print('Официальная конфигурация RealMLP_TD_CLASS загружена из DefaultParams и будет сохранена в result JSON.')


PyTabKit: 1.7.3; устройство: cpu
Выходные artifacts: stage11_realmlp_results_V1.json, stage11_realmlp_oof_V1.npz, stage11_realmlp_summary_V1.json
Официальная конфигурация RealMLP_TD_CLASS загружена из DefaultParams и будет сохранена в result JSON.


### Что проверяем?

Проверяем preflight до обучения: идентичность dataset и working indices, target/fold alignment, baseline GBDT_mean и набор ровно из 47 разрешённых features. Это непосредственно исключает leakage через validation fold и останавливает обучение при несоответствии принятому control. Никакая preprocessing information из outer-validation здесь не создаётся.


In [2]:
stage1 = json.loads(STAGE1_PATH.read_text(encoding='utf-8'))
stage7 = json.loads(STAGE7_PATH.read_text(encoding='utf-8'))
assert DATASET.exists() and STAGE7_OOF_PATH.exists(), 'STOP: отсутствует обязательный input artifact'
assert sha256_file(DATASET) == EXPECTED_DATASET_SHA, 'STOP: SHA dataset не совпадает'
features = list(stage1['допустимые_признаки'])
assert len(features) == 47 and len(set(features)) == 47, 'STOP: число или уникальность features не совпадают'
assert not set(FORBIDDEN).intersection(features), 'STOP: найден запрещённый predictor'
assert features == stage7['raw_features_in_order'], 'STOP: feature identity Stage 1/7 не совпадает'
assert stage7['baseline_selection']['B_star'] == 'GBDT_mean', 'STOP: принятый B* не совпадает'
assert stage7['dataset_sha256'] == EXPECTED_DATASET_SHA, 'STOP: SHA dataset Stage 7 не совпадает'
assert stage7['working_index_sha256'] == EXPECTED_WORKING_SHA, 'STOP: SHA working-index Stage 7 не совпадает'

raw = pd.read_excel(DATASET, engine='pyxlsb')
assert TARGET in raw and IDENTIFIER in raw, 'STOP: target или identifier отсутствует'
assert all(column in raw for column in features), 'STOP: отсутствует обязательный feature'
with np.load(STAGE7_OOF_PATH, allow_pickle=False) as artifact:
    required = {'working_indices', 'target', 'fold', 'gbdt_mean'}
    assert required.issubset(artifact.files), f'STOP: отсутствуют OOF keys: {required - set(artifact.files)}'
    working_indices = np.asarray(artifact['working_indices'], dtype=np.int64)
    y_working = np.asarray(artifact['target'], dtype=np.int8)
    fold = np.asarray(artifact['fold'], dtype=np.int8)
    gbdt_mean = np.asarray(artifact['gbdt_mean'], dtype=np.float64)

assert len(working_indices) == len(y_working) == len(fold) == len(gbdt_mean) == EXPECTED_N, 'STOP: неожиданное working n'
assert np.unique(working_indices).size == EXPECTED_N, 'STOP: дублируются working indices'
assert sha256_indices(working_indices) == EXPECTED_WORKING_SHA, 'STOP: SHA working-index не совпадает'
assert np.array_equal(raw.loc[working_indices, TARGET].to_numpy(dtype=np.int8), y_working), 'STOP: target alignment не совпадает'
assert np.isfinite(gbdt_mean).all() and ((0.0 <= gbdt_mean) & (gbdt_mean <= 1.0)).all(), 'STOP: GBDT_mean содержит невалидные значения'
assert set(np.unique(fold)) == {1, 2, 3}, 'STOP: невалидные fold labels'

splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
expected_fold = np.zeros(EXPECTED_N, dtype=np.int8)
for fold_id, (_, valid_pos) in enumerate(splitter.split(np.zeros(EXPECTED_N), y_working), start=1):
    expected_fold[valid_pos] = fold_id
assert np.array_equal(fold, expected_fold), 'STOP: сохранённое fold assignment отличается от locked outer CV'

X_working = raw.loc[working_indices, features].copy()
assert not X_working.isna().any().any(), 'STOP: RealMLP не принимает пропуски в numeric features; imputation здесь запрещён'
assert all(pd.api.types.is_numeric_dtype(X_working[column]) for column in features), 'STOP: обнаружен нечисловой feature'
assert np.isfinite(X_working.to_numpy(dtype=np.float64)).all(), 'STOP: feature содержит неfinite значение'
print(f'ПРЕДВАРИТЕЛЬНАЯ ПРОВЕРКА ПРОЙДЕНА: n={EXPECTED_N}, features={len(features)}, baseline finite, dataset/indices/target/folds согласованы.')


ПРЕДВАРИТЕЛЬНАЯ ПРОВЕРКА ПРОЙДЕНА: n=289614, features=47, baseline finite, dataset/indices/target/folds согласованы.


### Что проверяем?

Выполняем ровно три outer folds. Для каждого RealMLP получает только X_train/y_train; его официальная встроенная предобработка и внутренняя validation остаются fold-local. В notebook выводятся индикаторы по fold, стадии и elapsed time, а параметры конструктора сохраняются отдельно от эффективной преднастроенной конфигурации в result JSON. Это отвечает на вопрос сравнением свежего RealMLP OOF с уже сохранённым GBDT_mean без повторного GBDT run.

Полный запуск дорогой и запускается пользователем только после pre-run review.


In [3]:
import contextlib
import logging
import threading
import warnings

CHECKPOINT_PATH = GENERATED / 'stage11_realmlp_checkpoint_V1.npz'

_STAGE11_PANEL = None
_STAGE11_PANEL_LOCK = threading.Lock()
_STAGE11_PROGRESS_LOCK = threading.Lock()

_STAGE11_PROGRESS = {
    'fold_id': 1,
    'phase': 'подготовка',
    'fold_started': None,
    'last_delta_gini': None,
}


def format_runtime(seconds: float | None) -> str:
    """Человекочитаемое время для progress-panel."""
    if seconds is None:
        return '—'

    total = max(0, int(round(float(seconds))))
    hours, remainder = divmod(total, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours:
        return f'{hours} ч {minutes:02d} мин {secs:02d} сек'
    if minutes:
        return f'{minutes} мин {secs:02d} сек'
    return f'{secs} сек'


def checkpoint_contract() -> dict:
    """Поля, которые обязаны совпасть при resume."""
    return {
        'experiment': 'Stage 11',
        'version': 'V1',
        'dataset_sha256': EXPECTED_DATASET_SHA,
        'working_index_sha256': EXPECTED_WORKING_SHA,
        'raw_features_in_order': list(features),
        'outer_seed': OUTER_SEED,
        'fold_seeds': list(FOLD_SEEDS),
        'experiment_overrides': json_safe(EXPERIMENT_OVERRIDES),
        'pytabkit_version': importlib.metadata.version('pytabkit'),
    }


def fresh_stage11_state() -> dict:
    """Новое состояние Stage 11 без завершённых фолдов."""
    return {
        **checkpoint_contract(),
        'status': 'in_progress',
        'completed_folds': [],
        'fold_rows': [],
        'effective_configs': [],
        'runtime_seconds': 0.0,
        'active_fold': None,
        'phase': 'подготовка',
        'realmlp_oof': np.full(
            EXPECTED_N,
            np.nan,
            dtype=np.float64,
        ),
    }


def save_stage11_checkpoint(state: dict) -> None:
    """
    Атомарно сохраняем промежуточное состояние.

    Сам checkpoint не объявляет эксперимент completed.
    """
    metadata = {
        key: value
        for key, value in state.items()
        if key != 'realmlp_oof'
    }

    atomic_npz(
        CHECKPOINT_PATH,
        realmlp_oof=np.asarray(
            state['realmlp_oof'],
            dtype=np.float64,
        ),
        state_json=np.asarray(
            json.dumps(
                json_safe(metadata),
                ensure_ascii=False,
                allow_nan=False,
            )
        ),
    )


def load_stage11_checkpoint() -> tuple[dict, str]:
    """
    Загружаем checkpoint и проверяем его совместимость
    с текущим locked Stage 11.
    """
    if not CHECKPOINT_PATH.exists():
        state = fresh_stage11_state()
        save_stage11_checkpoint(state)
        return state, 'fresh'

    with np.load(
        CHECKPOINT_PATH,
        allow_pickle=False,
    ) as artifact:
        required = {
            'realmlp_oof',
            'state_json',
        }

        if not required.issubset(artifact.files):
            raise RuntimeError(
                'STOP: checkpoint Stage 11 имеет неожиданную структуру'
            )

        realmlp_oof = np.asarray(
            artifact['realmlp_oof'],
            dtype=np.float64,
        )

        metadata = json.loads(
            str(artifact['state_json'].item())
        )

    expected = checkpoint_contract()

    for key, expected_value in expected.items():
        if metadata.get(key) != expected_value:
            raise RuntimeError(
                f'STOP: checkpoint Stage 11 несовместим '
                f'по полю {key!r}'
            )

    if realmlp_oof.shape != (EXPECTED_N,):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит OOF '
            'неправильной длины'
        )

    completed_folds = {
        int(value)
        for value in metadata.get(
            'completed_folds',
            [],
        )
    }

    if not completed_folds.issubset({1, 2, 3}):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит неизвестный fold'
        )

    fold_rows = metadata.get(
        'fold_rows',
        [],
    )

    config_rows = metadata.get(
        'effective_configs',
        [],
    )

    row_folds = [
        int(row['fold'])
        for row in fold_rows
    ]

    config_folds = [
        int(row['fold'])
        for row in config_rows
    ]

    if (
        len(row_folds) != len(set(row_folds))
        or set(row_folds) != completed_folds
    ):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит '
            'несогласованные fold metrics'
        )

    if (
        len(config_folds) != len(set(config_folds))
        or set(config_folds) != completed_folds
    ):
        raise RuntimeError(
            'STOP: checkpoint Stage 11 содержит '
            'несогласованные configs'
        )

    for fold_id in (1, 2, 3):
        positions = fold == fold_id

        if fold_id in completed_folds:
            if not np.isfinite(
                realmlp_oof[positions]
            ).all():
                raise RuntimeError(
                    f'STOP: fold {fold_id} отмечен завершённым, '
                    'но его OOF заполнен не полностью'
                )

        else:
            # Незавершённый фолд никогда не используем частично.
            realmlp_oof[positions] = np.nan

    state = {
        **metadata,
        'completed_folds': sorted(
            completed_folds
        ),
        'realmlp_oof': realmlp_oof,
    }

    return state, 'resume'


def set_stage11_progress(**updates) -> None:
    """Совместимость с checkpoint/resume: live-панель отключена."""
    return None


def render_stage11_panel(
    state: dict,
    session_started: float,
    base_runtime: float,
) -> None:
    """Не показывает динамическую панель; прогресс выводится только на границах fold."""
    return None


def stage11_heartbeat(
    stop_event: threading.Event,
    state: dict,
    session_started: float,
    base_runtime: float,
) -> None:
    """Не печатает эпохи, локальные пути или технические сообщения."""
    return None


@contextlib.contextmanager
def suppress_lightning_technical_messages():
    """Временно оставляет для Lightning только ошибки."""
    logger_names = ('lightning', 'lightning.pytorch', 'pytorch_lightning')
    previous_levels = {name: logging.getLogger(name).level for name in logger_names}
    for name in logger_names:
        logging.getLogger(name).setLevel(logging.ERROR)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            category=Warning,
            module=r'^(?:lightning|pytorch_lightning)(?:\.|$)',
        )
        try:
            yield
        finally:
            for name, level in previous_levels.items():
                logging.getLogger(name).setLevel(level)


def fit_realmlp_quietly(
    model: RealMLP_TD_Classifier,
    x_train: pd.DataFrame,
    y_train: np.ndarray,
) -> None:
    """
    Запускаем тот же model.fit(), но не выводим в notebook
    технические логи PyTabKit и локальные пути.

    verbosity=2 в locked конфигурации НЕ меняется.
    """
    with tempfile.TemporaryFile(
        mode='w+',
        encoding='utf-8',
    ) as technical_output:

        with (
            suppress_lightning_technical_messages(),
            contextlib.redirect_stdout(technical_output),
            contextlib.redirect_stderr(technical_output),
        ):
            model.fit(
                x_train,
                y_train,
            )


def show_stage11_result(
    result: dict,
) -> None:
    """Короткий человекочитаемый итог вместо сырого JSON."""
    realmlp_metrics = (
        result['realmlp_oof_metrics']
    )

    baseline_metrics = (
        result['gbdt_mean_oof_metrics']
    )

    deltas = (
        result[
            'delta_realmlp_minus_gbdt_mean'
        ]
    )

    decision_text = {
        'material_gain':
            'существенное улучшение',
        'inferior':
            'RealMLP уступает эталонной модели GBDT_mean',
        'no_material_benefit':
            'материального преимущества нет',
    }.get(
        'зафиксировано служебное решение; подробности — в сохранённом JSON',
        result['decision'],
    )

    print('Этап 11 V1 завершён.')
    print(
        f"OOF Gini RealMLP:   "
        f"{realmlp_metrics['Gini']:.6f}"
    )
    print(
        f"OOF Gini GBDT_mean: "
        f"{baseline_metrics['Gini']:.6f}"
    )
    print(
        f"Разность Gini:      "
        f"{deltas['Gini']:+.6f}"
    )
    print(
        f"Решение:        "
        f"{decision_text} "
        f""
    )
    print(
        f"Общее время:    "
        f"{format_runtime(result['runtime_seconds'])}"
    )


def load_saved_stage11_result() -> dict:
    """Читает уже сохранённый итог без запуска ML."""
    if not RESULT_PATH.exists():
        raise RuntimeError('STOP: отсутствует сохранённый Stage 11 result')

    result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
    required = {'status', 'realmlp_oof_metrics', 'gbdt_mean_oof_metrics', 'delta_realmlp_minus_gbdt_mean', 'decision', 'runtime_seconds'}
    if not required.issubset(result):
        raise RuntimeError(f'STOP: сохранённый Stage 11 result неполный: {required - set(result)}')
    if result['status'] != 'completed':
        raise RuntimeError('STOP: сохранённый Stage 11 result не имеет completed status')
    return result


def render_saved_stage11_conclusion(result: dict) -> None:
    """Показывает разделы исследования из сохранённого JSON без ML."""
    from IPython.display import Markdown, display

    realmlp = result['realmlp_oof_metrics']
    baseline = result['gbdt_mean_oof_metrics']
    delta = result['delta_realmlp_minus_gbdt_mean']
    decision = result['decision']
    decision_text = {
        'material_gain': 'существенное улучшение',
        'inferior': 'RealMLP уступает эталонной модели GBDT_mean',
        'no_material_benefit': 'существенного преимущества нет',
    }.get(decision, 'зафиксировано служебное решение; подробности — в сохранённом JSON')
    interpretation = {
        'material_gain': 'RealMLP достиг заранее заданного существенного улучшения по основной OOF Gini.',
        'inferior': 'RealMLP существенно уступил GBDT_mean по заранее заданному правилу OOF Gini.',
        'no_material_benefit': 'RealMLP не показал существенного преимущества по заранее заданному правилу OOF Gini.',
    }.get(decision, 'В сохранённом результате указано служебное решение; подробности доступны в JSON-артефакте.')
    limitation_text = {
        'Random CV does not prove temporal stability.': 'Случайная кросс-валидация не подтверждает временную стабильность модели.',
        'Three folds are not a statistical-significance claim.': 'Три фолда не являются основанием для вывода о статистической значимости.',
        'Precision/Recall/F1 at 0.5 are diagnostic only.': 'Точность, полнота и F1 при пороге 0,5 используются только для диагностики.',
        'Final test was not used.': 'Финальный тест не использовался.',
        'One tuned-default RealMLP recipe was evaluated without KOMUS-specific HPO.': 'Оценён один рецепт RealMLP с настроенными значениями по умолчанию, без HPO, специфичного для KOMUS.',
    }
    limitations = '\n'.join(
        f"- {limitation_text.get(item, 'В сохранённом результате зафиксировано дополнительное ограничение; см. JSON-артефакт.')}"
        for item in result.get('limitations', [])
    ) or '- Ограничения в сохранённом результате не указаны.'

    display(Markdown(
        f'''# Результат исследования

## ФАКТЫ

- Источник: сохранённый stage11_realmlp_results_V1.json.
- RealMLP OOF Gini: **{realmlp['Gini']:.6f}**.
- GBDT_mean OOF Gini: **{baseline['Gini']:.6f}**.
- ΔGini: **{delta['Gini']:+.6f}**.
- Решение: **{decision_text}**.
- Длительность выполнения: **{format_runtime(result['runtime_seconds'])}**.

## ИНТЕРПРЕТАЦИЯ

{interpretation} Метрики точности, полноты и F1 при пороге 0,5 остаются только диагностическими и не изменяют решение.

## ОГРАНИЧЕНИЯ

{limitations}

## СЛЕДУЮЩИЙ ШАГ

Результат передаётся техническому координатору для решения о завершении или продолжении ветки, посвящённой только модели.'''
    ))


def decide(
    delta_gini: float,
    fold_deltas: list[float],
) -> str:
    """
    Исходное locked decision rule Stage 11.
    Логика не изменена.
    """
    wins = sum(
        delta > 0.0
        for delta in fold_deltas
    )

    losses = sum(
        delta < 0.0
        for delta in fold_deltas
    )

    if (
        delta_gini >= 0.010
        and wins >= 2
    ):
        return 'material_gain'

    if (
        delta_gini <= -0.010
        and losses >= 2
    ):
        return 'inferior'

    return 'no_material_benefit'


def run_stage11() -> dict:
    """
    Полный Stage 11 с fold-level checkpoint/resume.

    ML-протокол исходного эксперимента не меняется.
    """
    global _STAGE11_PANEL
    _STAGE11_PANEL = None

    # RESULT_PATH пишется последним.
    # Поэтому completed result защищает от повторного ML-run.
    if RESULT_PATH.exists():
        existing = json.loads(
            RESULT_PATH.read_text(
                encoding='utf-8'
            )
        )

        if existing.get('status') == 'completed':

            if (
                existing.get('dataset_sha256')
                != EXPECTED_DATASET_SHA
            ):
                raise RuntimeError(
                    'STOP: completed Stage 11 result '
                    'относится к другому dataset'
                )

            if (
                existing.get(
                    'working_index_sha256'
                )
                != EXPECTED_WORKING_SHA
            ):
                raise RuntimeError(
                    'STOP: completed Stage 11 result '
                    'относится к другой working sample'
                )

            if (
                not OOF_PATH.exists()
                or not SUMMARY_PATH.exists()
            ):
                raise RuntimeError(
                    'STOP: completed result найден, '
                    'но комплект Stage 11 artifacts неполный'
                )

            print(
                'Stage 11 уже завершён; '
                'повторный ML-run не запускается.'
            )

            show_stage11_result(existing)
            return existing

    state, launch_mode = (
        load_stage11_checkpoint()
    )

    session_started = (
        time.monotonic()
    )

    base_runtime = float(
        state.get(
            'runtime_seconds',
            0.0,
        )
    )

    finished = False

    set_stage11_progress(
        fold_id=1,
        phase=f'подготовка ({launch_mode})',
        fold_started=None,
        last_delta_gini=None,
    )

    render_stage11_panel(
        state,
        session_started,
        base_runtime,
    )

    stop_event = threading.Event()

    heartbeat = threading.Thread(
        target=stage11_heartbeat,
        args=(
            stop_event,
            state,
            session_started,
            base_runtime,
        ),
        daemon=True,
    )

    heartbeat.start()

    try:
        for fold_id, seed in enumerate(
            FOLD_SEEDS,
            start=1,
        ):
            completed = {
                int(value)
                for value
                in state['completed_folds']
            }

            if fold_id in completed:
                existing_row = next(
                    row
                    for row in state['fold_rows']
                    if int(row['fold']) == fold_id
                )

                set_stage11_progress(
                    fold_id=fold_id,
                    phase=(
                        'фолд уже завершён — '
                        'пропускаем'
                    ),
                    fold_started=None,
                    last_delta_gini=(
                        existing_row[
                            'delta_realmlp_minus_gbdt_mean'
                        ]['Gini']
                    ),
                )

                render_stage11_panel(
                    state,
                    session_started,
                    base_runtime,
                )

                continue

            train_pos = np.flatnonzero(
                fold != fold_id
            )

            valid_pos = np.flatnonzero(
                fold == fold_id
            )

            fold_started = (
                time.monotonic()
            )

            print(f'[Этап 11] Начало fold {fold_id}/3.')

            state['active_fold'] = fold_id
            state['phase'] = (
                'обучение RealMLP'
            )

            state['runtime_seconds'] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Checkpoint существует ещё до начала fit.
            save_stage11_checkpoint(state)

            set_stage11_progress(
                fold_id=fold_id,
                phase='обучение RealMLP',
                fold_started=fold_started,
                last_delta_gini=None,
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

            model = RealMLP_TD_Classifier(
                random_state=seed,
                **EXPERIMENT_OVERRIDES,
            )

            config_record = {
                'fold': fold_id,
                'seed': seed,
                'official_tuned_default_recipe':
                    copy.deepcopy(
                        OFFICIAL_REALMLP_TD_CLASS_DEFAULTS
                    ),
                'experiment_overrides': {
                    **EXPERIMENT_OVERRIDES,
                    'random_state': seed,
                },
                'effective_realmlp_config':
                    effective_realmlp_config(seed),
                'constructor_params':
                    json_safe(
                        model.get_params(
                            deep=False
                        )
                    ),
            }

            fit_realmlp_quietly(
                model,
                X_working.iloc[
                    train_pos
                ],
                y_working[
                    train_pos
                ],
            )

            state['phase'] = (
                'прогноз outer-validation'
            )

            set_stage11_progress(
                phase='прогноз outer-validation'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

            probability = np.asarray(
                model.predict_proba(
                    X_working.iloc[
                        valid_pos
                    ]
                )[:, 1],
                dtype=np.float64,
            )

            assert (
                np.isfinite(
                    probability
                ).all()
                and (
                    (
                        0.0
                        <= probability
                    )
                    & (
                        probability
                        <= 1.0
                    )
                ).all()
            ), (
                'STOP: RealMLP probability '
                'невалидна'
            )

            model_metrics = (
                metrics_at_threshold(
                    y_working[
                        valid_pos
                    ],
                    probability,
                )
            )

            baseline_metrics = (
                metrics_at_threshold(
                    y_working[
                        valid_pos
                    ],
                    gbdt_mean[
                        valid_pos
                    ],
                )
            )

            delta = {
                key:
                    model_metrics[key]
                    - baseline_metrics[key]
                for key
                in model_metrics
            }

            # Только после prediction + metrics
            # этот fold считается завершённым.
            state[
                'realmlp_oof'
            ][valid_pos] = probability

            state[
                'fold_rows'
            ].append(
                {
                    'fold': fold_id,
                    'seed': seed,
                    'n_train':
                        int(
                            train_pos.size
                        ),
                    'n_validation':
                        int(
                            valid_pos.size
                        ),
                    'runtime_seconds':
                        (
                            time.monotonic()
                            - fold_started
                        ),
                    'realmlp_metrics':
                        model_metrics,
                    'gbdt_mean_metrics':
                        baseline_metrics,
                    'delta_realmlp_minus_gbdt_mean':
                        delta,
                }
            )

            state[
                'effective_configs'
            ].append(
                config_record
            )

            state['completed_folds'] = (
                sorted(
                    completed
                    | {fold_id}
                )
            )

            state['active_fold'] = None
            state['phase'] = (
                'checkpoint сохранён'
            )

            state['runtime_seconds'] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Главный resumable checkpoint:
            # полностью законченный outer fold.
            save_stage11_checkpoint(state)

            completed_runtime = state['fold_rows'][-1]['runtime_seconds']
            print(f'[Этап 11] Завершение fold {fold_id}/3 | runtime: {format_runtime(completed_runtime)} | ΔGini: {delta["Gini"]:+.6f}')
            if fold_id == 1:
                print(f'[Этап 11] Приблизительно осталось: {format_runtime(2 * completed_runtime)} (по времени первого fold).')

            set_stage11_progress(
                fold_id=fold_id,
                phase=(
                    'фолд завершён; '
                    'checkpoint сохранён'
                ),
                fold_started=None,
                last_delta_gini=(
                    delta['Gini']
                ),
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        assert (
            set(
                int(value)
                for value
                in state[
                    'completed_folds'
                ]
            )
            == {1, 2, 3}
        ), (
            'STOP: не все outer folds '
            'завершены'
        )

        realmlp_oof = np.asarray(
            state['realmlp_oof'],
            dtype=np.float64,
        )

        assert np.isfinite(
            realmlp_oof
        ).all(), (
            'STOP: RealMLP OOF '
            'заполнен не полностью'
        )

        fold_rows = sorted(
            state['fold_rows'],
            key=lambda row:
                int(row['fold']),
        )

        effective_configs = sorted(
            state[
                'effective_configs'
            ],
            key=lambda row:
                int(row['fold']),
        )

        realmlp_metrics = (
            metrics_at_threshold(
                y_working,
                realmlp_oof,
            )
        )

        baseline_metrics = (
            metrics_at_threshold(
                y_working,
                gbdt_mean,
            )
        )

        deltas = {
            key:
                realmlp_metrics[key]
                - baseline_metrics[key]
            for key
            in realmlp_metrics
        }

        fold_gini_deltas = [
            row[
                'delta_realmlp_minus_gbdt_mean'
            ]['Gini']
            for row
            in fold_rows
        ]

        total_runtime = (
            base_runtime
            + (
                time.monotonic()
                - session_started
            )
        )

        result = {
            'experiment': 'Stage 11',
            'version': 'V1',
            'status': 'completed',
            'dataset_sha256':
                EXPECTED_DATASET_SHA,
            'working_index_sha256':
                EXPECTED_WORKING_SHA,
            'target': TARGET,
            'identifier': IDENTIFIER,
            'raw_features_in_order':
                features,
            'raw_feature_count':
                len(features),
            'forbidden_features':
                list(FORBIDDEN),
            'pytabkit_version':
                importlib.metadata.version(
                    'pytabkit'
                ),
            'model':
                'RealMLP_TD_Classifier',
            'official_tuned_default_recipe':
                OFFICIAL_REALMLP_TD_CLASS_DEFAULTS,
            'experiment_overrides':
                EXPERIMENT_OVERRIDES,
            'effective_realmlp_config_by_fold':
                effective_configs,
            'outer_cv': {
                'type':
                    'StratifiedKFold',
                'n_splits': 3,
                'shuffle': True,
                'random_state':
                    OUTER_SEED,
            },
            'outer_fold_seeds': {
                str(index): seed
                for index, seed
                in enumerate(
                    FOLD_SEEDS,
                    start=1,
                )
            },
            'baseline': {
                'name':
                    'GBDT_mean',
                'source':
                    str(
                        STAGE7_OOF_PATH
                        .relative_to(ROOT)
                    ),
                'retrained': False,
            },
            'protocol': {
                'device': 'cpu',
                'n_cv': 1,
                'n_refit': 0,
                'n_ens': 1,
                'hpo': False,
                'bagging_or_ensembling':
                    False,
                'calibration': False,
                'class_weighting':
                    False,
                'balancing_or_sampling':
                    False,
                'threshold_optimization':
                    False,
                'diagnostic_threshold':
                    THRESHOLD,
                'preprocessing':
                    (
                        'official RealMLP '
                        'native fold-local pipeline'
                    ),
            },
            'fold_metrics':
                fold_rows,
            'realmlp_oof_metrics':
                realmlp_metrics,
            'gbdt_mean_oof_metrics':
                baseline_metrics,
            'delta_realmlp_minus_gbdt_mean':
                deltas,
            'decision':
                decide(
                    deltas['Gini'],
                    fold_gini_deltas,
                ),
            'final_test_used':
                False,
            'runtime_seconds':
                total_runtime,
            'limitations': [
                (
                    'Random CV does not prove '
                    'temporal stability.'
                ),
                (
                    'Three folds are not a '
                    'statistical-significance claim.'
                ),
                (
                    'Precision/Recall/F1 at 0.5 '
                    'are diagnostic only.'
                ),
                (
                    'Final test was not used.'
                ),
                (
                    'One tuned-default RealMLP '
                    'recipe was evaluated without '
                    'KOMUS-specific HPO.'
                ),
            ],
        }

        summary_payload = {
            'experiment':
                'Stage 11 V1',
            'decision':
                result['decision'],
            'primary_metric':
                'OOF Gini',
            'realmlp_oof_gini':
                realmlp_metrics[
                    'Gini'
                ],
            'gbdt_mean_oof_gini':
                baseline_metrics[
                    'Gini'
                ],
            'delta_gini':
                deltas['Gini'],
            'fold_gini_deltas':
                fold_gini_deltas,
            'final_test_used':
                False,
            'result_artifacts': [
                str(
                    RESULT_PATH
                    .relative_to(ROOT)
                ),
                str(
                    OOF_PATH
                    .relative_to(ROOT)
                ),
            ],
        }

        # Важен порядок.
        # Completed RESULT_PATH создаётся последним.
        atomic_npz(
            OOF_PATH,
            working_indices=
                working_indices,
            target=
                y_working,
            fold=
                fold,
            gbdt_mean_probability=
                gbdt_mean,
            realmlp_oof_probability=
                realmlp_oof,
        )

        atomic_json(
            SUMMARY_PATH,
            summary_payload,
        )

        atomic_json(
            RESULT_PATH,
            result,
        )

        # Только после успешного сохранения
        # всего финального комплекта.
        CHECKPOINT_PATH.unlink(
            missing_ok=True
        )

        finished = True

        set_stage11_progress(
            fold_id=3,
            phase='эксперимент завершён',
            fold_started=None,
            last_delta_gini=
                deltas['Gini'],
        )

        render_stage11_panel(
            state,
            session_started,
            base_runtime,
        )

        show_stage11_result(
            result
        )

        return result

    except KeyboardInterrupt:
        if not finished:
            state['status'] = (
                'in_progress'
            )

            state['phase'] = (
                'остановлено пользователем'
            )

            state[
                'runtime_seconds'
            ] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            # Завершённые фолды сохраняются.
            # Незавершённый текущий fold
            # при resume будет выполнен заново.
            save_stage11_checkpoint(
                state
            )

            set_stage11_progress(
                phase=
                    'остановлено пользователем'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        raise

    except Exception:
        if not finished:
            state['status'] = (
                'in_progress'
            )

            state['phase'] = (
                'ошибка; checkpoint сохранён'
            )

            state[
                'runtime_seconds'
            ] = (
                base_runtime
                + (
                    time.monotonic()
                    - session_started
                )
            )

            save_stage11_checkpoint(
                state
            )

            set_stage11_progress(
                phase=
                    'ошибка; checkpoint сохранён'
            )

            render_stage11_panel(
                state,
                session_started,
                base_runtime,
            )

        raise

    finally:
        stop_event.set()
        heartbeat.join(
            timeout=2.0
        )

In [4]:
# Безопасный post-run вывод: читает готовый JSON и не запускает ML.
stage11_saved_result = load_saved_stage11_result()
show_stage11_result(stage11_saved_result)

Этап 11 V1 завершён.
OOF Gini RealMLP:   0.793325
OOF Gini GBDT_mean: 0.806399
Разность Gini:      -0.013074
Решение:        inferior 
Общее время:    2 ч 58 мин 50 сек


In [5]:
# Безопасный Markdown-итог: только сохранённый result JSON, без ML.
render_saved_stage11_conclusion(stage11_saved_result)

# Результат исследования

## ФАКТЫ

- Источник: сохранённый stage11_realmlp_results_V1.json.
- RealMLP OOF Gini: **0.793325**.
- GBDT_mean OOF Gini: **0.806399**.
- ΔGini: **-0.013074**.
- Решение: **RealMLP уступает эталонной модели GBDT_mean**.
- Длительность выполнения: **2 ч 58 мин 50 сек**.

## ИНТЕРПРЕТАЦИЯ

RealMLP существенно уступил GBDT_mean по заранее заданному правилу OOF Gini. Метрики точности, полноты и F1 при пороге 0,5 остаются только диагностическими и не изменяют решение.

## ОГРАНИЧЕНИЯ

- Случайная кросс-валидация не подтверждает временную стабильность модели.
- Три фолда не являются основанием для вывода о статистической значимости.
- Точность, полнота и F1 при пороге 0,5 используются только для диагностики.
- Финальный тест не использовался.
- Оценён один рецепт RealMLP с настроенными значениями по умолчанию, без HPO, специфичного для KOMUS.

## СЛЕДУЮЩИЙ ШАГ

Результат передаётся техническому координатору для решения о завершении или продолжении ветки, посвящённой только модели.

# Результат исследования

## ФАКТЫ

Stage 11 V1 завершён успешно.

Проверена одна заранее зафиксированная конфигурация `RealMLP_TD_Classifier` из PyTabKit на тех же 47 разрешённых признаках.

- использованы все `3 из 3` outer folds;
- рабочая выборка — `289 614` наблюдений;
- `GBDT_mean` заново не обучался и использовался как сохранённый Stage 7 OOF baseline;
- `Q_B1_norm` и `Q_B2_norm` не использовались как predictors;
- KOMUS-specific HPO, balancing, class weighting и threshold optimization не проводились;
- final test не использовался.

### Полный OOF

| Метрика | RealMLP | GBDT_mean | Δ RealMLP − baseline |
|---|---:|---:|---:|
| ROC-AUC | 0.896662 | 0.903200 | -0.006537 |
| Gini | **0.793325** | **0.806399** | **-0.013074** |
| PR-AUC | 0.587461 | 0.603857 | -0.016396 |
| Precision | 0.687871 | 0.724625 | -0.036755 |
| Recall | 0.376727 | 0.364162 | +0.012565 |
| F1 | 0.486831 | 0.484725 | +0.002106 |

### Gini по folds

| Fold | RealMLP | GBDT_mean | ΔGini |
|---:|---:|---:|---:|
| 1 | 0.782005 | 0.799785 | -0.017780 |
| 2 | 0.800068 | 0.807894 | -0.007826 |
| 3 | 0.799573 | 0.811695 | -0.012122 |

RealMLP уступил `GBDT_mean` по Gini на всех трёх folds.

Полный `ΔGini = -0.013074`.

Заранее зафиксированный статус `inferior` требовал:

- полного `ΔGini <= -0.010`;
- отрицательного направления минимум на `2 из 3` folds.

Оба условия выполнены.

**Decision: `inferior`.**

Полное время выполнения RealMLP experiment — `10 729.5 сек`, примерно `2 ч 59 мин` CPU-выполнения.

## ИНТЕРПРЕТАЦИЯ

В зафиксированном Stage 11 protocol RealMLP **не подтвердил преимущество над `GBDT_mean` и по основному правилу эксперимента оказался существенно хуже baseline**.

Разница наблюдается не только в полном OOF: Gini RealMLP ниже на каждом из трёх folds. Поэтому отрицательное направление результата устойчиво внутри использованного CV-протокола.

RealMLP также ниже по `ROC-AUC` и `PR-AUC`, то есть хуже ранжирует дефолтные и недефолтные компании в целом.

При диагностическом threshold `0.5` картина немного отличается:

- Recall RealMLP выше примерно на `1.26` п.п.;
- F1 выше примерно на `0.21` п.п.;
- одновременно Precision ниже примерно на `3.68` п.п.

Простыми словами, при этом конкретном cutoff RealMLP отмечает как рискованные немного больше дефолтных компаний, но одновременно делает больше ложных срабатываний.

Это не меняет исследовательский вывод. Порог `0.5` здесь не является выбранной рабочей точкой, а основной вопрос Stage 11 заранее решается по полному OOF Gini.

Таким образом, **ещё одна современная модель на том же наборе из 47 признаков не устранила наблюдаемый разрыв относительно сильного GBDT baseline**.

В контексте Stage 1–10 этот результат дополнительно поддерживает уже сложившуюся интерпретацию: текущие ограничения всё меньше похожи на проблему выбора конкретной архитектуры модели и всё больше — на ограничение доступной информации в 47-feature representation.

Это именно интерпретация совокупного evidence, а не доказательство того, что 47 признаков принципиально исчерпаны.

## ОГРАНИЧЕНИЯ

- Random CV не доказывает temporal stability.
- Три folds не являются доказательством statistical significance различий.
- `Precision`, `Recall` и `F1` при threshold `0.5` являются только диагностическими.
- Проверена одна официальная tuned-default конфигурация RealMLP без KOMUS-specific HPO. Результат не характеризует все возможные MLP-конфигурации.
- Experiment не доказывает business benefit или отсутствие business benefit.
- Результат не является причинным выводом о признаках или причинах дефолта.
- Final test не использовался.

## ВЫВОД

Исследовательский вопрос Stage 11 закрыт:

**официальный tuned-default RealMLP на тех же 47 разрешённых признаках не улучшил `GBDT_mean` и получил заранее определённый статус `inferior`.**

`Gini` снизился с `0.806399` до `0.793325`, то есть на `0.013074`, а отрицательное направление наблюдается на всех трёх folds.

RealMLP поэтому не даёт основания пересматривать принятый `GBDT_mean` как основной baseline текущей исследовательской ветки.

## СЛЕДУЮЩИЙ ЭТАП ИССЛЕДОВАНИЯ

После Stage 11 основной поиск новых моделей на текущих 47 разрешённых признаках рационально остановить.

Stage 1–11 уже проверили несколько разных гипотез о том, где может находиться резерв качества:

- сильные GBDT;
- современные tabular neural architectures;
- stacking;
- rank-complementarity;
- residual/oracle model reserve;
- независимый RealMLP.

Ни одна из этих веток не дала evidence, что основное ограничение текущего решения связано с недостаточно удачным выбором архитектуры модели.

Наоборот, совокупный результат всё сильнее поддерживает интерпретацию, что текущее ограничение скорее находится на уровне **доступной информации в 47-feature representation**.

Это не означает, что дальнейшее небольшое улучшение Gini математически невозможно. Это означает, что сейчас нет достаточного основания ожидать от очередной модели существенного нового знания для проекта.

### Решение

**Core model research на текущих 47 разрешённых признаках остановить.**

Зафиксировать research checkpoint:

`CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES`

Новый controlled model experiment пока не открывать.

### Следующий этап

Перейти к **Research Synthesis Stage 1–11** и собрать всю исследовательскую цепочку в единый защищаемый результат:

**47-feature baseline  
→ общая blind spot  
→ evidence missing signal  
→ современные model-only alternatives  
→ stacking  
→ complementarity diagnostics  
→ residual model reserve  
→ independent RealMLP confirmation  
→ обоснованный STOP model search.**

Итоговый synthesis должен включать:

- сводную таблицу результатов Stage 1–11;
- ключевые графики для презентации и защиты;
- связку `FACT → INTERPRETATION → LIMITATION`;
- dataset / split / feature / artifact identity;
- список закрытых, открытых и заблокированных вопросов;
- итоговый research conclusion;
- условия, при которых model research имеет смысл открыть снова.

### Когда исследование моделей можно возобновить

Возвращаться к model research рационально только при появлении нового evidence:

1. нового разрешённого источника признаков с понятным provenance;
2. надёжной row-level observation date или другого temporal anchor;
3. сильного независимого результата, который действительно противоречит текущему evidence;
4. конкретного business operating point по Recall / Precision / capacity / FN-FP policy;
5. новой исследовательской гипотезы, где заранее понятно, как положительный или отрицательный результат изменит решение проекта.

Само появление ещё одной новой архитектуры или библиотеки таким основанием не является.

**Следующий практический шаг проекта: synthesis → evidence/figures → presentation/defence → затем необходимые reusable foundations.**